<a href="https://colab.research.google.com/github/rdelhibabu/Q_DCNN-Bi-LSTM/blob/main/Q_DCNN%2BBi_LSTM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
# Cell 1: Install and Import Dependencies
!pip install qutip tensorflow numpy matplotlib scipy

import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, TimeDistributed, Bidirectional, LSTM, Layer
from tensorflow.keras.models import Model
import qutip as qt
import matplotlib.pyplot as plt
from scipy.integrate import odeint

print(f"TensorFlow Version: {tf.__version__}")
print(f"QuTiP Version: {qt.__version__}")

TensorFlow Version: 2.20.0
QuTiP Version: 5.3.0


In [17]:
# Cell 2: Physical Simulation Environment (QuTiP + MCWF)

# --- Physical Parameters (from Table 3) ---
w_q = 2 * np.pi * 5.10  # Qubit frequency (GHz)
w_c = 2 * np.pi * 6.85  # Cavity frequency (GHz)
chi = 2 * np.pi * 0.0025 # Dispersive shift (2.5 MHz)
kappa = 1.0             # Cavity decay rate (MHz)
gamma_1 = 0.5           # Qubit relaxation rate (MHz)
dt = 0.5                # Simulation step (ns)
T_sim = 1000            # Total simulation time steps

# --- Hamiltonian Setup ---
# Qubit operators
sm = qt.tensor(qt.identity(10), qt.destroy(2))
sz = qt.tensor(qt.identity(10), qt.sigmaz())
# Cavity operators
a = qt.tensor(qt.destroy(10), qt.identity(2))

# Dispersive Hamiltonian
H = w_c * a.dag() * a + 0.5 * w_q * sz + chi * a.dag() * a * sz

# Collapse operators for dissipation
c_ops = [np.sqrt(kappa) * a, np.sqrt(gamma_1) * sm]

# --- Colored Noise Generator (Ornstein-Uhlenbeck) ---
def generate_ou_noise(length, tau_c=50.0, dt=0.5, sigma=1.0):
    noise = np.zeros(length)
    decay = np.exp(-dt / tau_c)
    for i in range(1, length):
        noise[i] = noise[i-1] * decay + sigma * np.sqrt(1 - decay**2) * np.random.normal()
    return noise

# --- Trajectory Generator ---
def generate_trajectories(num_trajectories=100):
    """Generates stochastic MCWF trajectories and colored I-Q noise."""
    print(f"Generating {num_trajectories} continuous trajectories...")
    psi0 = qt.tensor(qt.basis(10, 0), qt.basis(2, 0)) # Initial state (Ground)
    tlist = np.linspace(0, T_sim * dt, T_sim)

    # Run Monte Carlo Solver with explicit keyword arguments for QuTiP 5 compatibility
    mc_result = qt.mcsolve(H, psi0, tlist, c_ops=c_ops, e_ops=[a.dag()*a, sz], ntraj=num_trajectories)

    I_Q_data = []
    phase_labels = [] # 0: Ground, 1: Excited, 2: Dissipation

    for i in range(num_trajectories):
        # Extract pure expectations
        n_cavity = mc_result.expect[0][i]
        sigma_z = mc_result.expect[1][i]

        # Inject colored noise to simulate heterodyne detection
        I_t = n_cavity * np.cos(chi * tlist) + generate_ou_noise(T_sim)
        Q_t = n_cavity * np.sin(chi * tlist) + generate_ou_noise(T_sim)

        I_Q_data.append(np.column_stack((I_t, Q_t)))

        # Derive simplified discrete ground truths based on sz
        labels = np.where(sigma_z < -0.9, 0, np.where(sigma_z > 0.9, 1, 2))
        phase_labels.append(labels)

    return np.array(I_Q_data), np.stack(phase_labels, axis=0)

# Generate a small test batch (increase num_trajectories for full dataset)
X_raw, Y_raw = generate_trajectories(num_trajectories=50)
print(f"Raw Data Shape: {X_raw.shape}")

Generating 50 continuous trajectories...
10.0%. Run time:   3.90s. Est. time left: 00:00:00:35
20.0%. Run time:   7.36s. Est. time left: 00:00:00:29
30.0%. Run time:  10.48s. Est. time left: 00:00:00:24
40.0%. Run time:  15.68s. Est. time left: 00:00:00:23
50.0%. Run time:  18.18s. Est. time left: 00:00:00:18
60.0%. Run time:  19.54s. Est. time left: 00:00:00:13
70.0%. Run time:  20.89s. Est. time left: 00:00:00:08
80.0%. Run time:  22.24s. Est. time left: 00:00:00:05
90.0%. Run time:  23.59s. Est. time left: 00:00:00:02
100.0%. Run time:  25.01s. Est. time left: 00:00:00:00
Total run time:  25.49s
Raw Data Shape: (50, 1000, 2)


In [18]:
# Cell 4: Hybrid Neural Architecture (Corrected Attention Layer)

from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, TimeDistributed, Bidirectional, LSTM, Layer
from tensorflow.keras.models import Model
import tensorflow as tf

class CustomAttention(Layer):
    """Global self-attention mechanism (Eq 12 & 13) - Dynamic Sequence Length Compliant."""
    def __init__(self, **kwargs):
        super(CustomAttention, self).__init__(**kwargs)

    def build(self, input_shape):
        # input_shape is (Batch, Time, Features). We only need the Feature dimension.
        feature_dim = input_shape[-1]

        # W_a maps the hidden states to a scalar attention score
        self.W_a = self.add_weight(name='attention_weight',
                                   shape=(feature_dim, 1),
                                   initializer='glorot_uniform', trainable=True)
        # Bias is a single scalar broadcasted across all time steps
        self.b_a = self.add_weight(name='attention_bias',
                                   shape=(1,),
                                   initializer='zeros', trainable=True)
        super(CustomAttention, self).build(input_shape)

    def call(self, x):
        # x shape: (Batch, Time, Features)
        # u_t = tanh(H_t * W_a + b_a) -> Resulting shape: (Batch, Time, 1)
        u_t = tf.nn.tanh(tf.tensordot(x, self.W_a, axes=1) + self.b_a)

        # alpha_t = softmax(u_t) across the Time axis (axis=1)
        alpha_t = tf.nn.softmax(u_t, axis=1)

        # Context vector V = sum(alpha_t * H_t)
        # We multiply the hidden states by their attention weights to pass forward
        weighted_output = x * alpha_t
        return weighted_output

def build_hybrid_model(input_shape=(None, 64, 64, 2), num_classes=3):
    """Constructs the DCNN + Bi-LSTM + Attention model."""
    inputs = Input(shape=input_shape, name="Spatial_Temporal_Tensor")

    # Spatial Extractor (DCNN) - Wrapped in TimeDistributed
    # Block 1
    x = TimeDistributed(Conv2D(16, (3, 3), activation='relu', padding='same'))(inputs)
    x = TimeDistributed(MaxPooling2D(pool_size=(2, 2)))(x)
    # Block 2
    x = TimeDistributed(Conv2D(32, (3, 3), activation='relu', padding='same'))(x)
    x = TimeDistributed(MaxPooling2D(pool_size=(2, 2)))(x)
    # Block 3
    x = TimeDistributed(Conv2D(64, (3, 3), activation='relu', padding='same'))(x)
    x = TimeDistributed(MaxPooling2D(pool_size=(2, 2)))(x)

    # Flatten spatial embeddings per time step
    x = TimeDistributed(Flatten())(x)

    # Dense compression to D_CNN = 256
    x = TimeDistributed(Dense(256, activation='linear', name="Spatial_Embeddings"))(x)

    # Temporal Processing (Bi-LSTM)
    x = Bidirectional(LSTM(256, return_sequences=True, dropout=0.2))(x)

    # Attention Weighting
    attention_output = CustomAttention(name="Attention_Layer")(x)

    # Discrete Phase Classifications (Softmax Output per time step)
    outputs = TimeDistributed(Dense(num_classes, activation='softmax', name="Phase_Classes"))(attention_output)

    model = Model(inputs=inputs, outputs=outputs)

    # Compile with L2 Tikhonov regularization applied via Adam weight decay
    optimizer = tf.keras.optimizers.Adam(learning_rate=5e-4, weight_decay=1e-4)
    model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])

    return model

# Instantiate and visualize the architecture
model = build_hybrid_model()
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ Spatial_Temporal_Tensor         │ (None, None, 64, 64,   │             0 │
│ (InputLayer)                    │ 2)                     │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_25             │ (None, None, 64, 64,   │           304 │
│ (TimeDistributed)               │ 16)                    │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_26             │ (None, None, 32, 32,   │             0 │
│ (TimeDistributed)               │ 16)                    │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_27             │ (None, None, 32, 32,   │         4,640 │
│ (TimeDistributed)               │ 32)                    │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_28             │ (None, None, 16, 16,   │             0 │
│ (TimeDistributed)               │ 32)                    │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_29             │ (None, None, 16, 16,   │        18,496 │
│ (TimeDistributed)               │ 64)                    │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_30             │ (None, None, 8, 8, 64) │             0 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_31             │ (None, None, 4096)     │             0 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_32             │ (None, None, 256)      │     1,048,832 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_3 (Bidirectional) │ (None, None, 512)      │     1,050,624 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Attention_Layer                 │ (None, None, 512)      │           513 │
│ (CustomAttention)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_33             │ (None, None, 3)        │         1,539 │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,124,948 (8.11 MB)

 Trainable params: 2,124,948 (8.11 MB)

 Non-trainable params: 0 (0.00 B)

In [4]:
# Fix Rank Mismatch
time_steps = X_tensor.shape[1]
Y_categorical_sequence = np.repeat(np.expand_dims(Y_categorical, axis=1), time_steps, axis=1)

In [6]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, TimeDistributed, Bidirectional, LSTM, Layer
from tensorflow.keras.models import Model
import qutip as qt

print("--- Phase 1: Physical Simulation ---")
w_q, w_c, chi = 2 * np.pi * 5.10, 2 * np.pi * 6.85, 2 * np.pi * 0.0025
kappa, gamma_1, dt, T_sim = 1.0, 0.5, 0.5, 1000
window_size, spatial_dim = 50, 64

sm = qt.tensor(qt.identity(10), qt.destroy(2))
sz = qt.tensor(qt.identity(10), qt.sigmaz())
a = qt.tensor(qt.destroy(10), qt.identity(2))
H = w_c * a.dag() * a + 0.5 * w_q * sz + chi * a.dag() * a * sz
c_ops = [np.sqrt(kappa) * a, np.sqrt(gamma_1) * sm]

def generate_ou_noise(length, tau_c=50.0, dt_sim=0.5, sigma=1.0):
    noise, decay = np.zeros(length), np.exp(-dt_sim / tau_c)
    for i in range(1, length):
        noise[i] = noise[i-1] * decay + sigma * np.sqrt(1 - decay**2) * np.random.normal()
    return noise

def generate_trajectories(num_trajectories=50):
    print(f"Generating {num_trajectories} stochastic trajectories...")
    psi0, tlist = qt.tensor(qt.basis(10, 0), qt.basis(2, 0)), np.linspace(0, T_sim * dt, T_sim)
    I_Q_data, phase_labels = [], []

    for _ in range(num_trajectories):
        mc_result = qt.mcsolve(H, psi0, tlist, c_ops=c_ops, e_ops=[a.dag()*a, sz], ntraj=1)
        n_cavity, sigma_z = np.array(mc_result.expect[0]).flatten(), np.array(mc_result.expect[1]).flatten()
        I_t = n_cavity * np.cos(chi * tlist) + generate_ou_noise(T_sim)
        Q_t = n_cavity * np.sin(chi * tlist) + generate_ou_noise(T_sim)
        I_Q_data.append(np.column_stack((I_t, Q_t)))
        phase_labels.append(np.where(sigma_z < -0.9, 0, np.where(sigma_z > 0.9, 1, 2)).astype(int))
    return np.array(I_Q_data), np.array(phase_labels)

X_raw, Y_raw = generate_trajectories(50)

print("\n--- Phase 2: Spatial-Temporal Tensorization ---")
def tensorize_trajectory(iq_data, s_dim=64, w_size=50):
    T, num_windows = iq_data.shape[0], iq_data.shape[0] // w_size
    tensor = np.zeros((num_windows, s_dim, s_dim, 2))
    amp_bins, time_steps, time_bins = np.linspace(-3, 3, s_dim + 1), np.arange(w_size), np.linspace(0, w_size, s_dim + 1)

    for w in range(num_windows):
        w_data = iq_data[w*w_size : (w+1)*w_size]
        H_I, _, _ = np.histogram2d(w_data[:, 0], time_steps, bins=[amp_bins, time_bins])
        H_Q, _, _ = np.histogram2d(w_data[:, 1], time_steps, bins=[amp_bins, time_bins])
        tensor[w, :, :, 0] = H_I / np.max(H_I + 1e-6)
        tensor[w, :, :, 1] = H_Q / np.max(H_Q + 1e-6)
    return tensor

X_tensors, Y_pooled = [], []
for i in range(X_raw.shape[0]):
    X_tensors.append(tensorize_trajectory(X_raw[i]))
    Y_pooled.append([np.bincount(Y_raw[i, w*50 : (w+1)*50]).argmax() for w in range(X_raw.shape[1] // 50)])

X_tensor = np.array(X_tensors)
# The labels are now perfectly shaped as (Batch, Time_Windows, Classes)
Y_categorical = tf.keras.utils.to_categorical(np.array(Y_pooled), num_classes=3)

print("\n--- Phase 3: Model Architecture ---")
class CustomAttention(Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def build(self, input_shape):
        self.W_a = self.add_weight(shape=(input_shape[-1], 1), initializer='glorot_uniform', trainable=True)
        self.b_a = self.add_weight(shape=(1,), initializer='zeros', trainable=True)
        # CRITICAL FIX: Tell Keras the layer is successfully built
        super().build(input_shape)

    def call(self, x):
        # Using tf.matmul is also safer for TF graph tracing than tensordot
        u_t = tf.nn.tanh(tf.matmul(x, self.W_a) + self.b_a)
        return x * tf.nn.softmax(u_t, axis=1)
inputs = Input(shape=(None, 64, 64, 2))
x = TimeDistributed(Conv2D(16, (3, 3), activation='relu', padding='same'))(inputs)
x = TimeDistributed(MaxPooling2D(pool_size=(2, 2)))(x)
x = TimeDistributed(Conv2D(32, (3, 3), activation='relu', padding='same'))(x)
x = TimeDistributed(MaxPooling2D(pool_size=(2, 2)))(x)
x = TimeDistributed(Conv2D(64, (3, 3), activation='relu', padding='same'))(x)
x = TimeDistributed(MaxPooling2D(pool_size=(2, 2)))(x)
x = TimeDistributed(Flatten())(x)
x = TimeDistributed(Dense(256, activation='linear'))(x)
x = Bidirectional(LSTM(256, return_sequences=True, dropout=0.2))(x)
attn_out = CustomAttention()(x)
outputs = TimeDistributed(Dense(3, activation='softmax'))(attn_out)

model = Model(inputs, outputs)
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=5e-4, weight_decay=1e-4),
              loss='categorical_crossentropy', metrics=['accuracy'])

print("\n--- Phase 4: Training & Inference ---")
# Feed Y_categorical directly into the target parameter
model.fit(X_tensor, Y_categorical, epochs=10, batch_size=8, validation_split=0.2, verbose=1)

predictions = model.predict(np.expand_dims(X_tensor[0], axis=0), verbose=0)
conf_scores, pred_classes = np.max(predictions, axis=-1), np.argmax(predictions, axis=-1)

print("\n--- Inference Output Snapshot (First 5 Windows) ---")
for t in range(5):
    state = f"State {pred_classes[0, t]} (Plateau)" if conf_scores[0, t] >= 0.72 else "Dissipation Burst (Ignore)"
    print(f"Temporal Window {t+1}: {state}")

--- Phase 1: Physical Simulation ---
Generating 50 stochastic trajectories...
100.0%. Run time:   0.00s. Est. time left: 00:00:00:00
Total run time:   0.28s
100.0%. Run time:   0.00s. Est. time left: 00:00:00:00
Total run time:   0.31s
100.0%. Run time:   0.00s. Est. time left: 00:00:00:00
Total run time:   0.27s
100.0%. Run time:   0.00s. Est. time left: 00:00:00:00
Total run time:   0.28s
100.0%. Run time:   0.00s. Est. time left: 00:00:00:00
Total run time:   0.27s
100.0%. Run time:   0.00s. Est. time left: 00:00:00:00
Total run time:   0.29s
100.0%. Run time:   0.00s. Est. time left: 00:00:00:00
Total run time:   0.26s
100.0%. Run time:   0.00s. Est. time left: 00:00:00:00
Total run time:   0.27s
100.0%. Run time:   0.00s. Est. time left: 00:00:00:00
Total run time:   0.28s
100.0%. Run time:   0.00s. Est. time left: 00:00:00:00
Total run time:   0.29s
100.0%. Run time:   0.00s. Est. time left: 00:00:00:00
Total run time:   0.27s
100.0%. Run time:   0.00s. Est. time left: 00:00:00:0